In [6]:
import pandas as pd
import numpy as np

# Semilla para reproducibilidad
np.random.seed(42)

# Generar 1000 registros sintéticos
n = 1000
edad = np.random.randint(18, 75, n)
saldo = np.random.uniform(0, 100000, n).round(2)
genero = np.random.choice(["Masculino", "Femenino"], n)
# Churn relacionado ligeramente con bajo saldo y edad joven
prob_churn = 1 / (1 + np.exp(-(0.03*(40 - edad) + 0.00003*(50000 - saldo))))
churn = np.random.binomial(1, prob_churn)

df = pd.DataFrame({
    "edad": edad,
    "saldo": saldo,
    "genero": genero,
    "churn": churn
})

df.to_csv("datos_churn.csv", index=False)
print("✅ Archivo 'datos_churn.csv' creado con éxito.")
df.head()
# Este código te generará un archivo datos_churn.csv directamente en tu entorno de Colab o local.



✅ Archivo 'datos_churn.csv' creado con éxito.


,edad,saldo,genero,churn
0,56,49261.81,Masculino,0
1,69,32875.16,Masculino,1
2,46,63340.09,Femenino,0
3,32,24014.56,Masculino,1
4,60,7586.33,Femenino,1


In [8]:
# Ejemplo de Código (Pseudocódigo PySpark):
#Python
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Instalar PySpark si estás en Colab
!pip install -q pyspark==3.5.1

# Crear sesión Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Modelo_Churn_Pipeline") \
    .getOrCreate()

print("✅ Sesión Spark inicializada correctamente.")

# 1. Cargar datos
df = spark.read.csv("datos_churn.csv", header=True, inferSchema=True)

# 2. Definir etapas del pipeline
# Transformador para convertir categoria string a indice numérico
indexer = StringIndexer(inputCol="genero", outputCol="genero_index")

# Transformador para One-Hot Encoding
encoder = OneHotEncoder(inputCol="genero_index", outputCol="genero_onehot")

# Ensamblador de características: combina todas las características en un solo vector
assembler = VectorAssembler(inputCols=["edad", "saldo", "genero_onehot"], outputCol="features")

# Escalador de características
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)

# Modelo de clasificación (Estimator)
lr = LogisticRegression(labelCol="churn", featuresCol="scaled_features")

# 3. Construir el Pipeline
pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, lr])

# 4. División de datos
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# 5. Entrenar el Pipeline (el Estimator dentro del Pipeline se entrena)
model = pipeline.fit(train_df)

# 6. Hacer predicciones en el conjunto de prueba
predictions = model.transform(test_df)

# 7. Evaluar el modelo
evaluator = BinaryClassificationEvaluator(labelCol="churn")
auc = evaluator.evaluate(predictions)
print(f"Área bajo la curva ROC (AUC): {auc}")

# 8. Opcional: Guardar el modelo del pipeline para uso futuro
# model.save("modelo_churn_pipeline")


✅ Sesión Spark inicializada correctamente.
Área bajo la curva ROC (AUC): 0.7487577639751559
